# EDA — Mi Spotify Wrapped (DWH Analysis)

**Proyecto**: dwh-spotify-wrapped  
**Materia**: Bases de Datos II — Universidad de Pamplona  
**Autor**: Didier  
**Fecha**: 2026-05-18

Análisis exploratorio del Data Warehouse dimensional construido sobre Cloud SQL (PostgreSQL 16).  
Todas las horas pico usan la columna `_cot` (Colombia, UTC-5) para reflejar el comportamiento real del oyente.

---
**Prerequisito**: subir `colab-eda-key.json` (Service Account `sa-colab-eda`) al runtime de Colab.

## 1. Setup — Dependencias y conexión

In [1]:
!pip install "cloud-sql-python-connector[pg8000]" SQLAlchemy pandas matplotlib seaborn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 3.5 MB/s eta 0:00:00


In [7]:
from google.colab import files
uploaded = files.upload()  # subir colab-eda-key.json

import os
key_file = list(uploaded.keys())[0]
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = f"/content/{key_file}"
print(f"Credenciales cargadas: {key_file}")

Saving colab-eda-key.json to colab-eda-key (2).json
Credenciales cargadas: colab-eda-key (2).json


In [10]:
from google.cloud.sql.connector import Connector
import sqlalchemy
import getpass

# ── Configuración ──────────────────────────────────────────────────────────
INSTANCE_CONNECTION_NAME = "dwh-spotify-wrapped:us-central1:spotify-postgres"
DB_USER     = "postgres"
DB_PASSWORD = getpass.getpass("DB password: ")   # ← se escribe a mano, no se guarda
DB_NAME     = "postgres"
# ──────────────────────────────────────────────────────────────────────────

connector = Connector()

def getconn():
    return connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pg8000",
        user=DB_USER,
        password=DB_PASSWORD,
        db=DB_NAME,
        ip_type="PUBLIC",
    )

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=getconn)
print("Conexión establecida con Cloud SQL")

DB password: ··········
Conexión establecida con Cloud SQL


## 2. Carga de datos del DWH

In [12]:
import pandas as pd

df_facts = pd.read_sql("""
    SELECT
        f.id AS history_id,
        f.played_at,
        f.hour_of_day,
        f.day_of_week,
        f.hour_of_day_cot,
        f.day_of_week_cot,
        f.context_type,
        t.name              AS track_name,
        t.lastfm_listeners  AS track_listeners,
        t.lastfm_playcount  AS track_playcount,
        t.duration_ms,
        t.explicit,
        a.name              AS artist_name,
        a.lastfm_listeners  AS artist_listeners,
        a.lastfm_tags       AS artist_tags
    FROM dwh.fact_listening_history f
    JOIN dwh.dim_tracks  t ON f.track_id  = t.track_id
    JOIN dwh.dim_artists a ON t.artist_id = a.artist_id
    ORDER BY f.played_at DESC
""", engine)

df_artists = pd.read_sql("SELECT * FROM dwh.dim_artists", engine)
df_tracks  = pd.read_sql("SELECT * FROM dwh.dim_tracks",  engine)
df_audit   = pd.read_sql("SELECT * FROM dwh.etl_audit ORDER BY started_at", engine)

print(f"Facts         : {len(df_facts):>5} filas")
print(f"Artistas      : {len(df_artists):>5} filas")
print(f"Canciones     : {len(df_tracks):>5} filas")
print(f"Runs ETL      : {len(df_audit):>5} registros")
df_facts.head()

Facts         :   123 filas
Artistas      :    69 filas
Canciones     :   132 filas
Runs ETL      :    16 registros


,history_id,played_at,hour_of_day,day_of_week,hour_of_day_cot,day_of_week_cot,context_type,track_name,track_listeners,track_playcount,duration_ms,explicit,artist_name,artist_listeners,artist_tags
0,109,2026-05-20 01:17:48.120,1,wednesday,20,tuesday,playlist,TU$$I (with Dei V),51636,814987,163200,True,YOVNGCHIMI,81571,"[drill, puerto rico, latin, rap, goat]"
1,110,2026-05-20 01:15:16.844,1,wednesday,20,tuesday,playlist,Ahora y Siempre,64650,958696,149275,False,Quevedo,391427,"[Reggaeton, rap, spanish, Hip-Hop, mediocre]"
2,111,2026-05-20 01:09:46.782,1,wednesday,20,tuesday,playlist,LO LOGRÉ,15698,102623,243893,True,Myke Towers,666582,"[Reggaeton, puerto rico, trap, latin, Hip-Hop]"
3,112,2026-05-20 01:00:16.848,1,wednesday,20,tuesday,playlist,Todo o Nada,35936,407242,159704,True,Eladio Carrion,406926,"[trap, latin, puerto rico, pop, electropop]"
4,113,2026-05-20 00:57:46.748,0,wednesday,19,tuesday,playlist,GOLDEN GUN,21000,313349,251560,False,Alvaro Diaz,267307,"[puerto rico, urban, rap, latin, rnb]"


## 3. ETL Audit — Historial de ejecuciones

In [13]:
audit_display = df_audit[[
    "started_at", "status", "history_new", "history_skipped",
    "artists_new", "tracks_new", "error_message"
]].copy()
audit_display["started_at"] = pd.to_datetime(audit_display["started_at"]).dt.strftime("%Y-%m-%d %H:%M")
print(f"Runs exitosos : {(df_audit.status == 'success').sum()}")
print(f"Total tracks ingresados: {df_audit.history_new.sum()}")
audit_display

Runs exitosos : 15
Total tracks ingresados: 123


,started_at,status,history_new,history_skipped,artists_new,tracks_new,error_message
0,2026-05-15 17:57,success,50,0,50,50,None
1,2026-05-15 18:20,success,0,49,0,0,None
2,2026-05-15 21:10,success,0,48,0,0,None
3,2026-05-18 21:57,success,0,0,0,0,None
4,2026-05-18 22:04,success,50,0,0,0,None
5,2026-05-18 22:16,success,0,0,0,0,None
6,2026-05-19 07:00,success,2,48,0,0,None
7,2026-05-19 18:18,success,6,0,0,0,None
8,2026-05-19 18:20,error,0,0,0,0,(psycopg2.errors.InFailedSqlTransaction) curre...
9,2026-05-19 18:31,success,0,0,0,0,None


# Observaciones:
el estudiante Didier (Data engineer) realizó la implementación del ETL el día 15 del mes actual con su Spotify asociado a el aplicativo, por ende el eda refleja su comportamiento de escucha